In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


data = pd.read_csv('games.csv')
data.columns
data.rename( columns={'Unnamed: 0':'index'}, inplace=True )

In [ ]:
data[data['Times Listed'] != data['Number of Reviews']]

In [ ]:
data = data.drop(columns='Times Listed')

In [ ]:
data = data.drop(columns='Reviews')

In [ ]:
data[data.duplicated(subset=["Title", "Release Date"])]

In [ ]:
data = data.drop_duplicates(subset=["Title", "Release Date"])

In [ ]:
data[data.Title == "Minecraft"]

In [ ]:
data = data.drop(columns='Team')

In [ ]:
data

In [ ]:
from datetime import datetime

date_format = "%b %d, %Y"

for x in data.index:
    date_string = data["Release Date"][x]
    if date_string == "releases on TBD":
        data = data.drop([x])
    else:
        parsed_date = datetime.strptime(date_string, date_format).date().year
        data.at[x, "Release Date"] = parsed_date


In [ ]:


data.Rating.unique()

In [ ]:


import math

data[(data.Rating).isna()]


In [ ]:
data = data.dropna(subset=["Rating"])
data

In [ ]:
data = data.drop(columns = 'Summary')

In [ ]:
data

In [ ]:
def convert_string_to_int(string):
    if string[-1] == "K":
        return int(float(string[:-1]) * 1000)
    else:
        return int(string)

for x in data.index:
    data["Number of Reviews"][x] = convert_string_to_int(data["Number of Reviews"][x])
    data.Plays[x] = convert_string_to_int(data.Plays[x])
    data.Playing[x] = convert_string_to_int(data.Playing[x])
    data.Backlogs[x] = convert_string_to_int(data.Backlogs[x])
    data.Wishlist[x] = convert_string_to_int(data.Wishlist[x])

In [ ]:
data

In [ ]:

data = data.assign(Popularity= data.Rating*( 0.1 * data["Number of Reviews"] + 0.8 * data.Plays + 1 * data.Playing + 0.3 * data.Backlogs + 0.2 * data.Wishlist))


In [ ]:
data

In [ ]:
from sklearn.preprocessing import OneHotEncoder
data = data.drop(columns = ['Number of Reviews','Plays', 'Playing', 'Backlogs', 'Wishlist', 'Rating'])





In [ ]:
data

In [ ]:
import numpy as np
popularity_bins = np.linspace(0, 130000, 11)

data['Popularity Class'] = pd.cut(data['Popularity'], bins=popularity_bins, labels=False, right=False)

data['Popularity Class'].unique()

In [ ]:
data['Words in title'] = data['Title'].apply(lambda x: len(x.split()))

data

In [ ]:
data['Release Date']

data = data.reset_index(drop=True)


In [ ]:
import tensorflow as tf
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

data = data.sample(frac = 1)


mlb = MultiLabelBinarizer()

genres_encoded = mlb.fit_transform(data['Genres'])

genres_encoded_df = pd.DataFrame(genres_encoded, columns=mlb.classes_)

data_encoded = pd.concat([data[['Words in title']],data[['Release Date']], genres_encoded_df], axis=1)


X = data_encoded.values

y = data['Popularity Class'].values




X_train, X_test = X[:400], X[400:]
y_train, y_test = y[:400], y[400:]

y_val, y_test = y_test[:400], y_test[400:]
X_val, X_test = X_test[:400], X_test[400:]

In [ ]:
import sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

np.random.seed(1)

seed = 123
kfold = StratifiedKFold(n_splits=5, random_state=seed, shuffle=True)

pipe_1 = Pipeline([('preprocessing', StandardScaler()), ('classifier', SVC())])
pipe_2 = Pipeline([('preprocessing', StandardScaler()), ('classifier', svm.LinearSVC())])
pipe_3 = Pipeline([('preprocessing', StandardScaler()), ('classifier', svm.SVC(kernel='poly'))])
pipe_4 = Pipeline([('preprocessing', StandardScaler()), ('classifier', LogisticRegression(max_iter=500))])

param_grid_1 = {'preprocessing': [StandardScaler(), None],
                'classifier__C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
                'classifier__gamma': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]}
param_grid_2 = {'preprocessing': [StandardScaler(), None],
                'classifier__C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]}
param_grid_3 = {'preprocessing': [StandardScaler(), None],
                'classifier__C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
                'classifier__degree': [1, 3, 7]}
print("READY")
grid_1 = GridSearchCV(pipe_1, param_grid_1, cv=kfold, return_train_score=True)
grid_1.fit(X_train, y_train)
print("OK1")
grid_2 = GridSearchCV(pipe_2, param_grid_2, cv=kfold, return_train_score=True)
grid_2.fit(X_train, y_train)
print("OK2")
grid_3 = GridSearchCV(pipe_3, param_grid_3, cv=kfold, return_train_score=True)
grid_3.fit(X_train, y_train)
print("OK3")
scorer = sklearn.metrics.make_scorer(sklearn.metrics.f1_score, average='weighted')

grid_4 = GridSearchCV(pipe_4, param_grid_2, scoring=scorer, cv=kfold, return_train_score=True)
grid_4.fit(X_train, y_train)

print("OK4")







In [ ]:
import pandas as pd

from keras.callbacks import EarlyStopping
from keras.layers import Dense
from keras import Sequential

X_train = tf.constant(X_train, dtype=tf.float32)
X_test = tf.constant(X_test, dtype=tf.float32)
X_val = tf.constant(X_val, dtype=tf.float32)
y_train = tf.constant(y_train, dtype=tf.float32)
y_test = tf.constant(y_test, dtype=tf.float32)
y_val = tf.constant(y_val, dtype=tf.float32)


print(y_val)
print(y_test)
print(y_train)

y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=10)
y_val = tf.keras.utils.to_categorical(y_val, num_classes=10)


modelNN = Sequential()
modelNN.add(Dense(100, activation="sigmoid", input_shape=(X_train.shape[1],)))
modelNN.add(Dense(50, activation="sigmoid"))
modelNN.add(Dense(20, activation="sigmoid"))
modelNN.add(Dense(10, activation="sigmoid"))
modelNN.summary()

modelNN.compile(loss="categorical_crossentropy", optimizer="Adam", metrics=["accuracy"])

early_stopping = EarlyStopping(monitor='val_loss', patience=7, mode='min', verbose=1)
modelNN.fit(X_train, y_train, validation_data=(X_val, y_val), batch_size=32, epochs=100, callbacks=[early_stopping])

evaluation = modelNN.evaluate(X_test, y_test)
print("Loss:", evaluation[0])
print("Accuracy:", evaluation[1])

In [ ]:

models = []
models.append(('SVM rbf', grid_1.best_estimator_))
models.append(('SVM linear', grid_2.best_estimator_))
models.append(('SVM poly', grid_3.best_estimator_))
models.append(('LogisticRegression', grid_4.best_estimator_))
models.append(('Neural Network', modelNN))

metrics_dict = {'precision_score': [],
                'recall_score': [],
                'f1_score': [],
                'accuracy_score': []}

y_test_classes = np.argmax(y_test, axis=1)

for name, model in models:
    print(name)
    if name == 'Neural Network':
        y_pred = np.argmax(model.predict(X_test), axis=1)
    else:
        y_pred = model.predict(X_test)

    precision = metrics.precision_score(y_test_classes, y_pred, average='weighted')
    recall = metrics.recall_score(y_test_classes, y_pred, average='weighted')
    f1 = metrics.f1_score(y_test_classes, y_pred, average='weighted')
    accuracy = metrics.accuracy_score(y_test_classes, y_pred)

    print("Precision score:", precision)
    print("Recall score:", recall)
    print("F1 score:", f1)
    print("Accuracy score:", accuracy)

    metrics_dict['precision_score'].append(precision)
    metrics_dict['recall_score'].append(recall)
    metrics_dict['f1_score'].append(f1)
    metrics_dict['accuracy_score'].append(accuracy)

df = pd.DataFrame(data=metrics_dict, index=['SVM rbf', 'SVM linear', 'SVM poly', 'Logistic Regression', 'Neural Network'])

df.plot(kind='bar')
plt.title('Comparison of Models')
plt.xlabel('Model')
plt.ylabel('Score')
plt.legend(loc='lower right')
plt.show()
